In [ ]:
# Load our libraries

## The ones we've used until now
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow import keras

## Some new ones
from sklearn.metrics import root_mean_squared_error
from tensorflow.keras.callbacks import EarlyStopping

## 1. Formulate / Outline the problem

We'll now shift focus to another dataset and another problem: Weather prediction. In specific, **predict sunshine hours for Basel**.

<img src="images/03_weather_prediction_dataset_map.png" width="800" />

Sunshine hours is a numeric variable, and therefore our problem is no longer classification (into categories) but rather **regression** (for numbers / continuous values).

## 2. Identify inputs and outputs

In [ ]:
# Load the data
data = pd.read_csv("https://zenodo.org/record/5071376/files/weather_prediction_dataset_light.csv?download=1")

In [ ]:
# Print some info for our data
print("Number of (rows, columns): ", data.shape)
print("Column names: ", data.columns)

In [ ]:
# Look at first five rows
data.head()

In [ ]:
# See what data types we have and if the dataset is complete
data.info()

Some things we can conclude from our dataset:
- 91 columns: date (yyymmdd format), month (numeric), and weather variables for 11 different cities in Europe.
- 3654 rows, meaning 3654 days of collected data (or 10 years).
- All variables/features are numeric
- It doesn't seem like we have empty/null values
- We want to predict `BASEL_sunshine` on day `i+1` based on all other variables on day `i`.

## 3. Prepare data

As usual, our data preparation requires: data cleaning, defining output/target/y and inputs/features/x, and splitting intro train and test sets

In [ ]:
## First, build the models as we only had information for 3 years
nr_rows = 365*3 # 3 years

# Prepare features. We won't be using DATE or MONTH
X_data = data.loc[0:nr_rows]
X_data = X_data.drop(columns=['DATE', 'MONTH'])

# Prepare target, knowing that our target is BASEL_sunshine for day i+1
y_data = data.loc[1:(nr_rows + 1), "BASEL_sunshine"]

### Data Splitting Strategy: Train, Validation, and Test

To build a model that actually works in the real world, we are moving from a 2-way split to a **3-way partition**.

Why the change?
Previously, we used the `test` set for evaluation. However, every time we "tweak" the model to get a better score on that test set, we are accidentally leaking information. The model is no longer being tested on "unseen" data because the developer has shaped the model to fit that specific data.

The Three Partitions

* **Train Set**: The "Textbook." This is the data the model looks at to learn patterns and adjust its weights.
* **Validation Set**: The "Practice Exam." We use this during training to compare different architectures and tune hyperparameters. The model doesn't learn from this data directly, but we use its results to decide which model version is best.
* **Test Set**: The "Final Exam." This is a locked vault. It is used **only once** at the very end to provide an unbiased measure of how the final model will perform in the real world.

In short:
| Split | Purpose | Used for Training? | Influences Model Design? |
| :--- | :--- | :--- | :--- |
| **Train** | Learning weights | Yes | Yes |
| **Validation** | Tuning & Selection | No | Indirectly (via the developer) |
| **Test** | Final Evaluation | No | No |

In [ ]:
# Split intro train and holdout (which will later be split into validation and test)
# A conventional 70% of the data for training
X_train, X_holdout, y_train, y_holdout =

# Using our holdout, split evenly (15/15%) into validation and test
X_val, X_test, y_val, y_test =

## 4. Build an architecture from scratch, or choose a pretrained model

We'll write a reusable function for an architecture with two hidden layers, but a variable number of: 1) inputs (features), 2) neurons in each layer.

Another key difference is that, as we now have a regression problem, we only have 1 output neuron.

In [ ]:
## Fill in the blanks

# Set our random seed
keras.utils.set_random_seed(2)

# Define our function for creating our NN model with varying hyperparameters

def create_nn(input_shape, n_neurons):
    # Input layer
    inputs =

    # Dense layers
    layers_dense = 
    layers_dense = 

    # Output layer
    outputs = 

    return keras.Model(inputs=inputs, outputs=outputs, name=f"weather_model_{n_neurons[0]}_{n_neurons[1]}")

In [ ]:
# Our starting model, with 100 neurons in the first hidden layer and 50 in the second one
model_100_50 = 

In [ ]:
# See our model summary


## How do models learn?

- Role of our loss function: quantify the total error of the predictions made by the model.
- During training (fitting) the model tries different weights/parameters, looking for those that minimize the loss
- It **doesn't look at random, it uses an algorith - Gradient descent**

### Gradient descent

Imagine we only had one neuron and were trying to find the optimal weight to minimize the loss.

The algorithm starts at a **random point**, and from there, using the gradient of the loss function, decides smartly where to take the next step (how to update the model weights). The size of the step that the algorithm takes is the **learning rate**.


![](images/03_gradient_descent.png)


### Batch gradient descent

As neural networks benefit from huge amounts of data, all of it is not fed to the model at once, but on **batches**.

The number of samples in one batch is called the **batch size**.

Each time the model has seen the entirity of our data, we call that an **epoch**.

### Questions

1. What is the goal of optimization?
    1. To find the weights that maximize the loss function
    1. To find the weights that minimize the loss function


2. What happens in one gradient descent step?
    1. The weights are adjusted so that we move in the direction of the gradient, so up the slope of the loss function
    1. The weights are adjusted so that we move in the direction of the gradient, so down the slope of the loss function
    1. The weights are adjusted so that we move in the direction of the negative gradient, so up the slope of the loss function
    1. The weights are adjusted so that we move in the direction of the negative gradient, so down the slope of the loss function

3. When the batch size is increased:(multiple answers might apply)
    1. The number of samples in an epoch also increases
    1. The number of batches in an epoch goes down
    1. The training progress is more jumpy, because more samples are consulted in each update step (one batch).
    1. The memory load (memory as in computer hardware) of the training process is increased

## 5. Choose a loss function and optimizer

As we already learned, we use `model.compile()` to specify our loss and optimizer.

Adding now:
- A function for compiling without retyping
- Mean Squared Error (MSE) as a loss function appropiate for regression
- A metric `Root Mean Squared Error (RMSE)` that we'll keep an eye during training, instead of just loss

In [ ]:
# Defining a function for compiling our model
def compile_model(model_obj):
    model_obj.compile(optimizer='adam',
                  loss='mse',
                  metrics=[keras.metrics.RootMeanSquaredError()])

In [ ]:
# Use the function for the model we already defined
compile_model(model_100_50)

## 6. Train the model

We'll use `model.fit()` with our training data, but now adding the `batch_size` argument

In [ ]:
## Fill in the blank

# Fitting our model to the training data
history_100_50 = 

In [ ]:
## Make a plot of our metrics over epochs

def plot_history(history, metrics):
    """
    Plot the training history

    Args:
        history (keras History object that is returned by model.fit())
        metrics (str, list): Metric or a list of metrics to plot
    """
    plt.style.use('ggplot')  # optional, that's only to define a visual style
    history_df = pd.DataFrame.from_dict(history.history)
    sns.lineplot(data=history_df[metrics])
    plt.xlabel("epochs")

In [ ]:
# Call the plot_history function we just created
plot_history(history_100_50, 'root_mean_squared_error')

## 7. Perform a prediction

In [ ]:
## Fill in the blanks

# With our trained model, predict sunshine hours for Basel on our train and test sets

y_train_predicted = 
y_val_predicted = 

## 8. Measure performance

As we had the confusion matrix for classification, we can do a **scatter plot** for regression, showing true vs predicted value

In [ ]:
# We define a function that we will reuse in this lesson
def plot_predictions(y_pred, y_true, title, ax):
    plt.style.use('ggplot')  # optional, that's only to define a visual style
    ax.scatter(y_pred, y_true, s=10, alpha=0.5)
    ax.axline((0,0),slope = 1, color = "black") # plot diagonal reference line
    ax.set_xlabel("predicted sunshine hours")
    ax.set_ylabel("true sunshine hours")
    ax.set_title(title)

In [ ]:
# Make a couple of plots, showing how our model performs in the data it used to learn (training set) and unseen data ()

fig, ax = plt.subplots(1,2)
plot_predictions(y_train_predicted, y_train, title='Predictions on the training set', ax = ax[0])
plot_predictions(y_val_predicted, y_val, title='Predictions on the test set', ax = ax[1])
plt.tight_layout()

As we defined a metric in our model, we can use `model.evaluate()` for the model to calculate the metric

In [ ]:
## Fill in the blanks

# Calculate metrics
train_metrics = 
val_metrics = 

# Print for comparison
print(f'Train RMSE: {train_metrics['root_mean_squared_error']:.2f}, Validation RMSE: {val_metrics['root_mean_squared_error']:.2f}')

#### Questions

- Is our model good?
- How can we say if our model is good?
- What is our goal here: That the model can learn the data perfectly, or that it's able to generalize to unseen data?

### Setting a baseline

It's hard to say if your model is doing a good job if you don't have a baseline to compare.

In our case, some basic baseline comparisons are:
- Predict the same sunshine hours than the day before
- Predict the average of the last x days

In [ ]:
# Create the baseline as the same sunshine hours than the day before
y_baseline_prediction = X_val['BASEL_sunshine']

# Calculate the RMSE for the baseline
rmse_baseline = root_mean_squared_error(y_val, y_baseline_prediction)
print('Baseline:', rmse_baseline)
# Compare to our model's RMSE
print('Neural network: ', val_metrics['root_mean_squared_error'])

Here we have a good sign, our model has a lower error (RMSE), which means it predicts better than the baseline.

### Overfitting

When building a model, one thing we want to avoid is overfitting.

It happens when performance in our validation set is much worse than in the train set. Basically, our model is able to learn from the training data, but not able to generalize to unseen data.

To check for overfitting, let's add the `validation_data` argument to the `model.fit()` function, and see how performance evolves as we train our network more (increased epochs).

In [ ]:
# Starting over for Keras not to continue training where it finished last time
model_100_50 = create_nn(input_shape=(X_data.shape[1],), 
                         n_neurons = [100,50])
compile_model(model_100_50)

# Adding our validation_data argument
history_100_50 = model_100_50.fit(X_train, y_train,
                    batch_size=32,
                    epochs=200,
                    validation_data=(X_val, y_val))

In [ ]:
# Plotting the RMSE for training and validation data
plot_history(history_100_50, ['root_mean_squared_error', 'val_root_mean_squared_error'])

#### Question:
Is it a good sign that our validation RMSE increases while the training RMS decreases?

## 9. Refine the model

Try a different architecture by:
- reusing the create_nn function and changing the number of neurons or the number of features included
- changing the number of layers, or activation function in hidden layers
- adding a `BatchNormalization` layer ([documentation here](https://keras.io/api/layers/normalization_layers/batch_normalization/)) right after the input layer

Answer these questions:
- How does your new model compare to our baseline and our first model?
- How does it relate in terms of overfitting compared to our first model?

In [ ]:
# ---------- Your Code Here------------------

# -------------------------------------------

### Early Stopping:

A technique to avoid overfitting. Basically, tells the models to stop training when the validation loss hasn't improved in the last n epochs.

In [ ]:
## Fill in the blanks 

# Define architecture
model_50_50 = create_nn(input_shape=(X_data.shape[1],), 
                         n_neurons = [50,50])

# Compile with loss function and optimizer
compile_model(model_50_50)

# Setting early stopping parameters
earlystopper =

# Training
history_50_50 = model_50_50.fit(X_train, y_train,
                    batch_size=32,
                    epochs=200,
                    ,
                    )

In [ ]:
plot_history(history_50_50, ['root_mean_squared_error', 'val_root_mean_squared_error'])